In [2]:
#Find the data files
import os 
for root,_,files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root,f))

/kaggle/input/datasets/shayanfazeli/heartbeat/ptbdb_abnormal.csv
/kaggle/input/datasets/shayanfazeli/heartbeat/ptbdb_normal.csv
/kaggle/input/datasets/shayanfazeli/heartbeat/mitbih_test.csv
/kaggle/input/datasets/shayanfazeli/heartbeat/mitbih_train.csv


In [3]:
#Data loading & tensor making
import pandas as pd, numpy as np, torch
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset,DataLoader
TRAIN_PATH="/kaggle/input/datasets/shayanfazeli/heartbeat/mitbih_train.csv"
TEST_PATH="/kaggle/input/datasets/shayanfazeli/heartbeat/mitbih_test.csv"

train=pd.read_csv(TRAIN_PATH,header=None)
test=pd.read_csv(TEST_PATH,header=None)

X=train.iloc[:,:187].values.astype(np.float32)
y=train[187].values.astype(np.int64)
X_test=test.iloc[:,:187].values.astype(np.float32)
y_test=test[187].values.astype(np.int64)

#Holding 10% of the train set as validation set(stratified keeps class ratios equal)
X_tr,X_val,y_tr,y_val=train_test_split(X,y,test_size=0.1,stratify=y,random_state=42)
    
def to_loader(X,y,shuffle):
    X_t=torch.from_numpy(X).unsqueeze(1) #(N,187) -> (N,1,187)
    y_t=torch.from_numpy(y)
    return DataLoader(TensorDataset(X_t,y_t),batch_size=128,shuffle=shuffle)
train_loader=to_loader(X_tr,y_tr,shuffle=True)
val_loader=to_loader(X_val,y_val,shuffle=True)
test_loader=to_loader(X_test,y_test,shuffle=True)

xb,yb=next(iter(train_loader))
print(xb.shape,yb.shape)

torch.Size([128, 1, 187]) torch.Size([128])


In [4]:
#define the CNN
import torch.nn as nn

class ECGCNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, padding=2),  nn.ReLU(), nn.MaxPool1d(2),   # 187 -> 93
            nn.Conv1d(16, 32, kernel_size=5, padding=2), nn.ReLU(), nn.MaxPool1d(2),   # 93 -> 46
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.ReLU(), nn.MaxPool1d(2),   # 46 -> 23
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                    # (64 channels * 23 steps) = 1472 values
            nn.Linear(64 * 23, 64), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),      # 5 scores, one per heartbeat class
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = ECGCNN()
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

ECGCNN(
  (features): Sequential(
    (0): Conv1d(1, 16, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(16, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): ReLU()
    (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (7): ReLU()
    (8): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1472, out_features=64, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=64, out_features=5, bias=True)
  )
)
Parameters: 107589


In [5]:
#shape check with a real batch
model.eval()
with torch.no_grad():
    out = model(xb)
print(out.shape)

torch.Size([128, 5])
